# 3. Cross-Modal Hypothesis Grounding

Align three information modalities in a shared embedding space:
1. **Text**: natural language hypothesis ("COOH* forms on Cu(111)")
2. **Graph**: reaction network structure (intermediates, steps, TS edges)
3. **Property**: free energy profile (ΔG values from DFT)

Uses **InfoNCE contrastive loss** (CLIP-style) for training and **cosine similarity** for inference scoring.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 3.1 Define a reaction network

In [ ]:
from science.alignment.hypothesis_grounder import (
    HypothesisGrounder, ReactionNetwork, ReactionStep,
    TextEncoder, ReactionGraphEncoder, FreeEnergyEncoder,
)

# CO2RR carboxyl pathway on Cu(111)
co2rr_data = {
    'reaction_network': [
        {'lhs': ['CO2(g)', '*', 'H+', 'e-'], 'rhs': ['COOH*']},
        {'lhs': ['COOH*', 'H+', 'e-'], 'rhs': ['CO*', 'H2O(g)']},
        {'lhs': ['CO*'], 'rhs': ['CO(g)', '*']},
    ],
    'intermediates': ['*', 'CO2(g)', 'COOH*', 'CO*', 'CO(g)', 'H2O(g)'],
    'ts_edges': [['CO2*', 'COOH*'], ['COOH*', 'CO*']],
    'coads_pairs': [['CO*', 'H*']],
    'surface': 'Cu(111)', 'reactant': 'CO2', 'product': 'CO',
}
network = ReactionNetwork.from_dict(co2rr_data)
print(f'Network: {len(network.steps)} steps, {len(network.intermediates)} intermediates')
print(f'Fingerprint: {network.fingerprint()}')

## 3.2 Encode each modality independently

In [ ]:
grounder = HypothesisGrounder(d_embed=64, temperature=0.07)

# Encode text hypothesis
hypothesis = 'COOH* is the rate-limiting intermediate for CO2 reduction to CO on Cu(111) via the carboxyl pathway'
t_emb = grounder.text_enc.encode(hypothesis)
print(f'Text embedding: shape={t_emb.shape}, norm={np.linalg.norm(t_emb):.3f}')

# Encode reaction graph
g_emb = grounder.graph_enc.encode(network)
print(f'Graph embedding: shape={g_emb.shape}, norm={np.linalg.norm(g_emb):.3f}')

# Encode free energy profile
dG = [0.0, 0.22, -0.15, -0.45, -1.10]  # CO2RR literature values
p_emb = grounder.prop_enc.encode(dG, U_limiting=-0.72, overpotential=0.61)
print(f'Property embedding: shape={p_emb.shape}, norm={np.linalg.norm(p_emb):.3f}')

## 3.3 Compute cross-modal alignment score
Score = sigmoid(weighted cosine similarity / temperature)

In [ ]:
# Full score with all 3 modalities
score = grounder.score(
    hypothesis=hypothesis,
    network=network,
    dG_profile=dG,
    U_limiting=-0.72,
    overpotential=0.61,
)
print(f'Alignment score: {score:.4f} (0=inconsistent, 1=fully aligned)')

# Detailed breakdown
breakdown = grounder.score_breakdown(hypothesis, network, dG)
print(f'\nScore breakdown:')
for k, v in breakdown.items():
    print(f'  {k:35s} = {v:.4f}')

## 3.4 Compare different hypotheses
A correct hypothesis should score higher than incorrect ones.

In [ ]:
hypotheses = [
    ('Correct: CO2RR carboxyl on Cu(111)',
     'The carboxyl pathway COOH* intermediate forms on Cu(111) during CO2 reduction to CO'),
    ('Wrong surface: CO2RR on Pt(111)',
     'Platinum Pt(111) surface catalyses CO2 reduction via formate HCOO* intermediate'),
    ('Wrong reaction: HER on Cu(111)',
     'Hydrogen evolution reaction on Cu(111) via Volmer-Heyrovsky mechanism'),
    ('Vague: generic catalysis',
     'Some catalytic reaction occurs on a metal surface with adsorbed species'),
]

scores = []
for label, hyp in hypotheses:
    s = grounder.score(hyp, network, dG)
    scores.append(s)
    print(f'{s:.4f}  {label}')

plt.figure(figsize=(8, 4))
colors = ['green', 'orange', 'red', 'gray']
plt.barh(range(len(scores)), scores, color=colors, alpha=0.8)
plt.yticks(range(len(scores)), [h[0] for h in hypotheses])
plt.xlabel('Alignment Score')
plt.title('Cross-modal alignment: correct hypothesis should score highest')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## 3.5 InfoNCE Contrastive Loss (Training)
The loss pulls positive (text, graph) pairs together and pushes negatives apart.

In [ ]:
# Create a small batch of (hypothesis, network) pairs
her_data = {
    'reaction_network': [{'lhs': ['H+', 'e-', '*'], 'rhs': ['H*']},
                         {'lhs': ['H*', 'H+', 'e-'], 'rhs': ['H2(g)', '*']}],
    'intermediates': ['*', 'H+', 'e-', 'H*', 'H2(g)'],
    'surface': 'Pt(111)', 'reactant': 'H+', 'product': 'H2',
}
oer_data = {
    'reaction_network': [{'lhs': ['H2O(g)', '*'], 'rhs': ['OH*', 'H+', 'e-']},
                         {'lhs': ['OH*'], 'rhs': ['O*', 'H+', 'e-']},
                         {'lhs': ['O*', 'H2O(g)'], 'rhs': ['OOH*', 'H+', 'e-']}],
    'intermediates': ['*', 'OH*', 'O*', 'OOH*', 'O2(g)', 'H2O(g)'],
    'surface': 'IrO2(110)', 'reactant': 'H2O', 'product': 'O2',
}

batch_texts = [
    hypothesis,
    'Hydrogen evolution on Pt(111) via Volmer-Heyrovsky mechanism',
    'Oxygen evolution on IrO2(110) lattice oxygen mechanism',
]
batch_networks = [
    network,
    ReactionNetwork.from_dict(her_data),
    ReactionNetwork.from_dict(oer_data),
]

loss = grounder.infonce_loss(batch_texts, batch_networks)
print(f'InfoNCE loss: {loss:.4f}')
print(f'(Lower = better alignment between positive pairs)')